# Laboratório — Regressão logística e classificação probabilística

Este laboratório implementa a regressão logística binária do zero, valida seu gradiente, compara-a ao scikit-learn e separa a probabilidade estimada da política de decisão.

[Voltar para a aula](../aulas/06-regressao-logistica-classificacao-probabilistica.md)

## TL;DR

- dados Bernoulli sintéticos tornam conhecido o processo gerador;
- o teste é isolado antes de qualquer seleção;
- `C` é selecionado por validação cruzada apenas no treino;
- o limiar é escolhido por custo na validação;
- modelo e política congelados são avaliados uma única vez no teste;
- a cópia versionada permanece sem outputs após a execução de validação.

## Contexto, dependências e protocolo

**Unidade de análise:** uma transação sintética independente. **Evento positivo:** ocorrência do evento de risco. **Seed:** `20260908`.

Dependências mínimas: Python 3.10, NumPy 1.24, pandas 2.0, Matplotlib 3.7 e scikit-learn 1.4. Não há download, credencial nem dado pessoal.

In [ ]:
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    brier_score_loss,
    confusion_matrix,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260908
np.set_printoptions(precision=6, suppress=True)

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)

## 1. Dados com probabilidade conhecida

Geramos seis features latentes, incluindo duas correlacionadas. Depois alteramos suas unidades em até quatro ordens de grandeza. A classe é amostrada de uma Bernoulli cuja probabilidade vem de um logit linear. Isso não torna a tarefa trivial: há incerteza aleatória mesmo quando o modelo está corretamente especificado.

In [ ]:
rng = np.random.default_rng(SEED)
n = 1_800
u = rng.normal(size=(n, 5))
X_base = np.column_stack([
    u[:, 0],
    u[:, 1],
    u[:, 2],
    u[:, 3],
    0.82 * u[:, 0] + 0.25 * rng.normal(size=n),
    u[:, 4],
])
scales = np.array([1.0, 100.0, 0.01, 5.0, 1.0, 20.0])
X = X_base * scales
feature_names = ["frequencia", "valor_centavos", "tempo_dias", "desvio", "sinal_redundante", "volume"]

beta_true = np.array([1.15, -0.85, 0.75, 0.55, 0.35, -0.60])
true_logit = -0.55 + X_base @ beta_true
true_probability = 1 / (1 + np.exp(-true_logit))
y = rng.binomial(1, true_probability)

assert X.shape == (1_800, 6)
assert set(np.unique(y)) == {0, 1}
print("Shape:", X.shape)
print("Prevalência observada:", f"{y.mean():.6f}")
print("Probabilidade geradora média:", f"{true_probability.mean():.6f}")
print("Correlação entre features redundantes:", f"{np.corrcoef(X_base[:, 0], X_base[:, 4])[0, 1]:.6f}")
print("Razão de escalas:", f"{X.std(axis=0).max() / X.std(axis=0).min():.1f}×")

## 2. Desenvolvimento, treino, validação e teste

Separamos 25% para teste antes de qualquer escolha. O desenvolvimento restante é dividido em treino e validação. A validação cruzada do hiperparâmetro usa somente o treino; a validação escolhe a política de limiar; o teste permanece intocado.

In [ ]:
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED + 1
)

assert len(X_train) + len(X_val) == len(X_dev)
assert len(X_dev) + len(X_test) == n
print("Treino:", X_train.shape, "| validação:", X_val.shape, "| teste:", X_test.shape)
print("Prevalências:", np.round([y_train.mean(), y_val.mean(), y_test.mean()], 6))

## 3. Sigmoide e BCE numericamente estáveis

`stable_sigmoid` evita calcular exponencial positiva enorme. A BCE parte dos logits com `np.logaddexp`, de modo que continua finita até para valores extremos.

In [ ]:
def stable_sigmoid(z):
    z = np.asarray(z, dtype=float)
    result = np.empty_like(z)
    positive = z >= 0
    result[positive] = 1 / (1 + np.exp(-z[positive]))
    exp_z = np.exp(z[~positive])
    result[~positive] = exp_z / (1 + exp_z)
    return result


def bce_from_logits(logits, target):
    logits = np.asarray(logits, dtype=float)
    target = np.asarray(target, dtype=float)
    return np.mean(np.logaddexp(0.0, logits) - target * logits)


extreme_logits = np.array([-1_000.0, -4.0, 0.0, 4.0, 1_000.0])
extreme_probabilities = stable_sigmoid(extreme_logits)
extreme_loss = bce_from_logits(extreme_logits, np.array([0, 0, 1, 1, 1]))
print("Sigmoide:", extreme_probabilities)
print("BCE finita com logits extremos:", f"{extreme_loss:.12f}")

assert np.isfinite(extreme_probabilities).all()
assert np.isfinite(extreme_loss)
assert stable_sigmoid(np.array([0.0]))[0] == 0.5

## 4. Loss, gradiente e verificação por diferenças finitas

O scaler é ajustado apenas no treino. Implementamos BCE média e os gradientes de pesos e intercepto. Uma diferença central aproxima numericamente cada derivada.

In [ ]:
scaler_scratch = StandardScaler().fit(X_train)
Z_train = scaler_scratch.transform(X_train)
Z_val = scaler_scratch.transform(X_val)


def loss_and_gradient(weights, intercept, features, target):
    logits = features @ weights + intercept
    probabilities = stable_sigmoid(logits)
    error = probabilities - target
    loss = bce_from_logits(logits, target)
    grad_w = features.T @ error / len(target)
    grad_b = error.mean()
    return loss, grad_w, grad_b


probe_w = rng.normal(0, 0.2, Z_train.shape[1])
probe_b = -0.1
loss_probe, analytic_w, analytic_b = loss_and_gradient(probe_w, probe_b, Z_train[:250], y_train[:250])
h = 1e-5
numeric_w = np.zeros_like(probe_w)
for j in range(len(probe_w)):
    direction = np.zeros_like(probe_w)
    direction[j] = h
    plus = loss_and_gradient(probe_w + direction, probe_b, Z_train[:250], y_train[:250])[0]
    minus = loss_and_gradient(probe_w - direction, probe_b, Z_train[:250], y_train[:250])[0]
    numeric_w[j] = (plus - minus) / (2 * h)
plus_b = loss_and_gradient(probe_w, probe_b + h, Z_train[:250], y_train[:250])[0]
minus_b = loss_and_gradient(probe_w, probe_b - h, Z_train[:250], y_train[:250])[0]
numeric_b = (plus_b - minus_b) / (2 * h)

max_gradient_error = max(np.max(np.abs(analytic_w - numeric_w)), abs(analytic_b - numeric_b))
print("Loss no ponto de teste:", f"{loss_probe:.9f}")
print("Erro máximo do gradient check:", f"{max_gradient_error:.3e}")
assert max_gradient_error < 1e-8

## 5. Treinamento do zero

Usamos gradiente descendente em dados padronizados. Este ajuste sem penalidade serve para validar o mecanismo matemático; o modelo operacional será regularizado e selecionado depois.

In [ ]:
weights = np.zeros(Z_train.shape[1])
intercept = 0.0
learning_rate = 0.25
loss_history = []

for step in range(8_000):
    loss, grad_w, grad_b = loss_and_gradient(weights, intercept, Z_train, y_train)
    weights -= learning_rate * grad_w
    intercept -= learning_rate * grad_b
    if step % 40 == 0:
        loss_history.append(loss)

final_loss, final_grad_w, final_grad_b = loss_and_gradient(weights, intercept, Z_train, y_train)
final_gradient_norm = np.sqrt(np.sum(final_grad_w**2) + final_grad_b**2)
print("Loss inicial/final:", f"{loss_history[0]:.9f}", f"{final_loss:.9f}")
print("Norma do gradiente final:", f"{final_gradient_norm:.3e}")

assert final_loss < loss_history[0]
assert final_gradient_norm < 1e-7

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(np.arange(len(loss_history)) * 40, loss_history, color="#2563eb")
ax.set(xlabel="Iteração", ylabel="BCE de treino", title="Convergência do gradiente descendente")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

**Texto alternativo:** curva da binary cross-entropy decresce rapidamente nas primeiras iterações e se estabiliza perto do mínimo, sem oscilações ou divergência.

## 6. Comparação com scikit-learn

Usamos `C=10^8` para aproximar o ajuste sem penalização sem depender de parâmetros que mudaram entre versões. Comparamos coeficientes e probabilidades na mesma representação padronizada.

In [ ]:
sk_unregularized = LogisticRegression(
    C=1e8, solver="lbfgs", max_iter=10_000, tol=1e-12, random_state=SEED
).fit(Z_train, y_train)

scratch_probability = stable_sigmoid(Z_val @ weights + intercept)
sk_probability = sk_unregularized.predict_proba(Z_val)[:, 1]
coefficient_difference = np.max(np.abs(weights - sk_unregularized.coef_[0]))
probability_difference = np.max(np.abs(scratch_probability - sk_probability))

print("Diferença máxima de coeficientes:", f"{coefficient_difference:.3e}")
print("Diferença máxima de probabilidades:", f"{probability_difference:.3e}")
print("Log-loss na validação (do zero):", f"{log_loss(y_val, scratch_probability):.9f}")

assert coefficient_difference < 2e-4
assert probability_difference < 5e-5

## 7. Seleção da regularização dentro do treino

O pipeline reaprende o scaler em cada fold. Selecionamos `C` por log-loss média em validação cruzada estratificada. Em `LogisticRegression`, `C` maior representa regularização mais fraca.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
candidates = np.logspace(-2, 2, 9)
cv_rows = []

for c_value in candidates:
    candidate = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=c_value, solver="lbfgs", max_iter=3_000, random_state=SEED),
    )
    fold_loss = -cross_val_score(
        candidate, X_train, y_train, scoring="neg_log_loss", cv=cv, n_jobs=1
    )
    cv_rows.append({
        "C": c_value,
        "log_loss_cv": fold_loss.mean(),
        "erro_padrao": fold_loss.std(ddof=1) / np.sqrt(len(fold_loss)),
    })

cv_results = pd.DataFrame(cv_rows)
best_c = float(cv_results.loc[cv_results["log_loss_cv"].idxmin(), "C"])
print(cv_results.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("C selecionado:", best_c)

assert best_c in candidates

## 8. Probabilidades na validação e interpretação dos coeficientes

O modelo com `C` selecionado é ajustado no treino. Como as features foram padronizadas, cada razão de odds abaixo corresponde ao aumento de um desvio-padrão da feature no treino, mantendo as demais constantes.

In [ ]:
selected_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=best_c, solver="lbfgs", max_iter=3_000, random_state=SEED),
).fit(X_train, y_train)
val_probability = selected_model.predict_proba(X_val)[:, 1]

coefficient_table = pd.DataFrame({
    "feature": feature_names,
    "coeficiente_padronizado": selected_model[-1].coef_[0],
})
coefficient_table["razao_de_odds"] = np.exp(coefficient_table["coeficiente_padronizado"])
coefficient_table["magnitude"] = coefficient_table["coeficiente_padronizado"].abs()
print(coefficient_table.sort_values("magnitude", ascending=False).drop(columns="magnitude").to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("\nLog-loss de validação:", f"{log_loss(y_val, val_probability):.6f}")
print("ROC AUC de validação:", f"{roc_auc_score(y_val, val_probability):.6f}")

## 9. Limiar escolhido por custo, não pelo teste

Definimos custo 5 para falso negativo e 1 para falso positivo. Verdadeiros positivos e negativos têm custo zero nesta simplificação. Avaliamos a grade apenas na validação.

In [ ]:
COST_FP = 1.0
COST_FN = 5.0
thresholds = np.linspace(0.05, 0.95, 91)


def decision_summary(target, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(target, prediction, labels=[0, 1]).ravel()
    return {
        "limiar": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "custo_medio": (COST_FP * fp + COST_FN * fn) / len(target),
        "precision": precision_score(target, prediction, zero_division=0),
        "recall": recall_score(target, prediction, zero_division=0),
    }


threshold_table = pd.DataFrame([
    decision_summary(y_val, val_probability, threshold) for threshold in thresholds
])
best_threshold = float(threshold_table.loc[threshold_table["custo_medio"].idxmin(), "limiar"])
comparison_val = pd.DataFrame([
    decision_summary(y_val, val_probability, 0.5),
    decision_summary(y_val, val_probability, best_threshold),
])
print(comparison_val.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("Limiar congelado:", f"{best_threshold:.2f}")

assert 0.05 < best_threshold < 0.95
assert comparison_val.iloc[1]["custo_medio"] <= comparison_val.iloc[0]["custo_medio"]

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
ax.plot(threshold_table["limiar"], threshold_table["custo_medio"], color="#dc2626")
ax.axvline(best_threshold, color="black", linestyle="--", label=f"selecionado = {best_threshold:.2f}")
ax.set(xlabel="Limiar", ylabel="Custo médio na validação", title="Política de decisão definida sem consultar o teste")
ax.grid(alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

**Texto alternativo:** curva do custo médio em função do limiar; a linha vertical marca o menor custo observado na validação. Limiares altos aumentam falsos negativos, que custam cinco vezes mais neste cenário.

## 10. Reajuste no desenvolvimento e avaliação final única

Com `C` e limiar congelados, reajustamos o pipeline em todo o desenvolvimento. Só agora calculamos probabilidades no teste. A prevalência do desenvolvimento é o baseline probabilístico constante.

In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=best_c, solver="lbfgs", max_iter=3_000, random_state=SEED),
).fit(X_dev, y_dev)

test_probability = final_model.predict_proba(X_test)[:, 1]
baseline_probability = np.full(len(y_test), y_dev.mean())
test_default = decision_summary(y_test, test_probability, 0.5)
test_selected = decision_summary(y_test, test_probability, best_threshold)

probability_metrics = pd.DataFrame([
    {
        "modelo": "Baseline prevalência",
        "log_loss": log_loss(y_test, baseline_probability),
        "brier": brier_score_loss(y_test, baseline_probability),
        "roc_auc": 0.5,
    },
    {
        "modelo": "Regressão logística",
        "log_loss": log_loss(y_test, test_probability),
        "brier": brier_score_loss(y_test, test_probability),
        "roc_auc": roc_auc_score(y_test, test_probability),
    },
])
decision_metrics = pd.DataFrame([test_default, test_selected])

print("Métricas probabilísticas no teste:")
print(probability_metrics.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
print("\nPolíticas no teste:")
print(decision_metrics.to_string(index=False, float_format=lambda value: f"{value:.6f}"))

assert probability_metrics.iloc[1]["log_loss"] < probability_metrics.iloc[0]["log_loss"]
assert probability_metrics.iloc[1]["roc_auc"] > 0.70
assert np.isfinite(test_probability).all()

## 11. Diagnóstico de confiabilidade

A curva agrupa probabilidades de teste em dez faixas de mesma largura. Ela é diagnóstica: bins com poucas observações são ruidosos e este único teste não substitui monitoramento após implantação.

In [ ]:
observed_frequency, mean_probability = calibration_curve(
    y_test, test_probability, n_bins=10, strategy="uniform"
)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
axes[0].plot([0, 1], [0, 1], "--", color="black", label="calibração ideal")
axes[0].plot(mean_probability, observed_frequency, marker="o", color="#2563eb", label="modelo")
axes[0].set(xlabel="Probabilidade média prevista", ylabel="Frequência positiva observada", title="Diagrama de confiabilidade")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].hist(test_probability, bins=np.linspace(0, 1, 11), color="#7c3aed", edgecolor="white")
axes[1].set(xlabel="Probabilidade prevista", ylabel="Número de casos", title="Distribuição das probabilidades")
plt.tight_layout()
plt.show()

print("Pares probabilidade/frequência:")
print(pd.DataFrame({"probabilidade_media": mean_probability, "frequencia_observada": observed_frequency}).round(6).to_string(index=False))

**Texto alternativo:** à esquerda, pontos de frequência observada são comparados à diagonal de calibração perfeita; à direita, um histograma mostra como as probabilidades previstas se distribuem entre zero e um.

## 12. Casos confiantes e errados

Inspecionar erros evita reduzir o experimento a uma média. Como os rótulos foram amostrados de uma Bernoulli, mesmo uma probabilidade geradora correta pode resultar em um evento improvável.

In [ ]:
test_prediction = (test_probability >= best_threshold).astype(int)
error_frame = pd.DataFrame({
    "y": y_test,
    "probabilidade": test_probability,
    "classe": test_prediction,
})
error_frame["confiança_errada"] = np.where(
    error_frame["y"] == 1,
    1 - error_frame["probabilidade"],
    error_frame["probabilidade"],
)
wrong = error_frame[error_frame["y"] != error_frame["classe"]].nlargest(8, "confiança_errada")
print(wrong.to_string(index=False, float_format=lambda value: f"{value:.6f}"))
assert len(wrong) > 0

## 13. Verificações finais e limites

Os testes abaixo confirmam o protocolo executado. Eles não provam causalidade, validade para outro domínio nem estabilidade sob mudança de prevalência.

In [ ]:
checks = {
    "seed_fixa": SEED,
    "teste_separado_antes_das_escolhas": True,
    "scaler_dentro_do_pipeline_operacional": True,
    "C_escolhido_no_treino": True,
    "limiar_escolhido_na_validacao": True,
    "gradient_check": max_gradient_error < 1e-8,
    "convergencia_do_zero": final_gradient_norm < 1e-7,
    "outputs_versionados": "limpos após validar uma cópia",
}
assert all(value for key, value in checks.items() if isinstance(value, bool))
print(pd.Series(checks).to_string())

## Takeaways

1. Logits são números reais; a sigmoide os transforma em probabilidades.
2. A BCE Bernoulli pode ser calculada de forma estável diretamente dos logits.
3. O gradiente analítico coincide com diferenças finitas e com o ajuste de referência.
4. Regularização e scaler pertencem ao pipeline e são escolhidos sem consultar o teste.
5. Probabilidade e política de decisão são separadas.
6. Custos assimétricos podem justificar um limiar diferente de 0,5.
7. Ranking não garante calibração; probabilidades devem ser diagnosticadas e monitoradas.

Na Aula 07, compare este modelo global com o KNN, que decide pela vizinhança local e torna a geometria da representação ainda mais explícita.